# Uji Statistik Parametrik
Dalam dunia industri maupun akademik, pengambilan keputusan berbasis data tidak lepas dari penerapan uji statistik. Uji parametrik, yang berlandaskan asumsi distribusi tertentu (seperti normalitas dan homogenitas varians), menjadi salah satu metode utama dalam menganalisis data kuantitatif.

Portofolio ini disusun dengan tujuan untuk mendemonstrasikan penerapan berbagai uji parametrik menggunakan Python dan dataset publik dari pustaka Seaborn. Dataset yang digunakan antara lain:
1. Tips Dataset → untuk analisis tip pelanggan restoran.
2. Iris Dataset → untuk menguji perbedaan rata-rata antar spesies bunga.
3. Penguins Dataset → untuk mengukur hubungan dan perbedaan morfologi antar spesies penguin.

Uji-uji statistik yang ditampilkan meliputi:
1. Uji Asumsi: Normalitas (Shapiro–Wilk) dan Homogenitas Varians (Levene’s Test).
2. Uji t-Test: One-Sample, Independent Samples, dan Paired Samples.
3. ANOVA (One-Way): untuk perbedaan rata-rata lebih dari dua kelompok.
4. Korelasi Pearson: untuk menguji hubungan linear antar variabel kuantitatif.
5. MANOVA (Multivariate ANOVA): untuk menguji perbedaan rata-rata multivariat antar kelompok.
6. Uji Chi-Square: untuk menguji distribusi data kategorikal atau kesesuaian frekuensi observasi dengan distribusi harapan.

Dengan penjelasan yang terstruktur dan interpretasi hasil yang mendalam, portofolio ini tidak hanya menampilkan kode dan output, tetapi juga narasi analitis yang dapat mendukung pemahaman dalam konteks data science, riset pasar, hingga industri.

# Import Library & Dataset

In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.multivariate.manova import MANOVA

# Load datasets
tips = sns.load_dataset("tips")
iris = sns.load_dataset("iris")
penguins = sns.load_dataset("penguins")

In [ ]:
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [ ]:
iris.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [ ]:
penguins.isnull().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


In [ ]:
penguins.dropna(inplace=True)

In [ ]:
penguins.isnull().sum()

,0
species,0
island,0
bill_length_mm,0
bill_depth_mm,0
flipper_length_mm,0
body_mass_g,0
sex,0


# Uji Hubungan Kategorikal (Chi-Square Test)
* Pertanyaan Penelitian\
Apakah terdapat hubungan signifikan antara panjang sepal (pendek vs panjang) dengan spesies bunga iris?
* Untuk menjawab pertanyaan ini, kita menggunakan uji Chi-Square independensi, yang cocok untuk data kategorikal.
* Hipotesis:
  
  H₀: distribusi panjang sepal tidak bergantung pada spesies (tidak ada hubungan).

  H₁: distribusi panjang sepal bergantung pada spesies (ada hubungan).

Binning sepal_length jadi kategori

In [11]:
iris["sepal_cat"] = pd.cut(iris["sepal_length"],
                           bins=[0, 5.5, 8],
                           labels=["Short", "Long"])

Buat tabel kontingensi

In [12]:
contingency = pd.crosstab(iris["sepal_cat"], iris["species"])
print(contingency)

species    setosa  versicolor  virginica
sepal_cat                               
Short          47          11          1
Long            3          39         49


Uji Chi-Square

In [20]:
chi2, p, dof, expected = stats.chi2_contingency(contingency)
print("Expected:\n", np.round(expected,2))
print("Chi-square:", np.round(chi2,2))
print("p-value:", p)
print("Degrees of freedom:", dof)

Expected:
 [[19.67 19.67 19.67]
 [30.33 30.33 30.33]]
Chi-square: 98.12
p-value: 4.940452295286229e-22
Degrees of freedom: 2


In [23]:
# χ² kritis
alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, dof)
print("Chi-square critical value:", round(chi2_critical,2))

Chi-square critical value: 5.99


* Hasil Uji Chi-Square\
  Chi-Square statistic (χ²) = 98.12\
  Derajat bebas (df) = 2\
  p-value = 4.94e-22\
  χ² kritis (α = 0.05, df = 2) ≈ 5.99
* Interpretasi
  * Karena χ² hitung = 98.12 lebih besar daripada χ² kritis = 5.99, maka tolak H₀.
  * Dengan p < 0.05, hasil ini juga mendukung penolakan H₀.
  * Artinya: terdapat hubungan signifikan antara panjang sepal dan spesies bunga iris. Dengan kata lain, distribusi panjang sepal berbeda nyata antar spesies.
* Kesimpulan\
Uji Chi-Square menunjukkan bahwa spesies bunga iris berpengaruh signifikan terhadap kategori panjang sepal. Temuan ini masuk akal karena setiap spesies iris memang memiliki karakteristik morfologi yang berbeda.

# Uji Asumsi (Normalitas & Homogenitas)
* Pertanyaan Penelitian\
Apakah terdapat perbedaan rata-rata tip yang diberikan oleh pelanggan pria dan wanita?
* Sebelum melanjutkan ke uji t atau ANOVA, ada dua asumsi utama yang harus dipenuhi:
  1. Normalitas → data dalam tiap kelompok harus berdistribusi normal.
  2. Homogenitas varians → varians antar kelompok yang dibandingkan harus sama.
* Pada contoh ini, diuji distribusi tip berdasarkan jenis kelamin pelanggan (Male dan Female) dari dataset tips.

In [ ]:
male_tips = tips[tips["sex"] == "Male"]["tip"]
female_tips = tips[tips["sex"] == "Female"]["tip"]

## Uji normalitas Shapiro-Wilk

In [ ]:
print("Shapiro-Wilk Male:", stats.shapiro(male_tips))
print("Shapiro-Wilk Female:", stats.shapiro(female_tips))

Shapiro-Wilk Male: ShapiroResult(statistic=np.float64(0.8758690617789388), pvalue=np.float64(3.7084828294513925e-10))
Shapiro-Wilk Female: ShapiroResult(statistic=np.float64(0.9567775372726819), pvalue=np.float64(0.005448280473692281))


* Hasil Uji Normalitas (Shapiro–Wilk Test)\
Male: statistic = 0.876, p-value = 3.71e-10\
Female: statistic = 0.957, p-value = 0.0054
* Interpretasi
  - H₀: data berasal dari distribusi normal.
  - Karena p < 0.05 untuk kedua kelompok → tolak H₀.
  - Artinya: baik tip pria maupun tip wanita tidak berdistribusi normal.

## Uji homogenitas varians (Levene's test)

In [ ]:
print("Levene test:", stats.levene(male_tips, female_tips))

Levene test: LeveneResult(statistic=np.float64(1.9909710178779405), pvalue=np.float64(0.1595236359896614))


In [ ]:
# Derajat kebebasan
k = 2  # jumlah kelompok
n_total = len(male_tips) + len(female_tips)
df_between = k - 1
df_within = n_total - k

# F-tabel pada alpha = 0.05
f_table = stats.f.ppf(1-0.05, df_between, df_within)

print("F-kritis: ",f_table)

F-kritis:  3.8801716626689986


* Hasil Uji Homogenitas Varians (Levene’s Test)\
F-statistic = 1.991\
F-kritis = 3.88\
p-value = 0.160
* Interpretasi:
  * H₀: varians antar kelompok sama (homogen).
  * Karena F-statistic = 1.991 lebih kecil daripada F-kritis = 3.88, maka gagal menolak H₀.
  * Karena p > 0.05 → gagal menolak H₀.
  * Artinya: varians tip pria dan wanita dapat dianggap homogen.

* Kesimpulan
  * Asumsi normalitas tidak terpenuhi.
  * Asumsi homogenitas varians terpenuhi.
  * Karena normalitas gagal, uji parametrik (t-test) tetap mungkin digunakan jika sampel besar berkat Teorema Limit Pusat.
  * Namun, untuk hasil yang lebih robust, disarankan menggunakan uji non-parametrik (Mann–Whitney U test).

# One-Sample t-test
* Pertanyaan Penelitian\
Apakah rata-rata tip yang diberikan pelanggan berbeda signifikan dari $3?

* Uji Parametrik: One-Sample t-test
* Tujuan: Menguji apakah rata-rata sampel berbeda signifikan dari suatu nilai tertentu (µ₀).
* Hipotesis:

  H₀: μ = 3 (rata-rata tip sama dengan $3)

  H₁: μ ≠ 3 (rata-rata tip berbeda dari $3)

In [ ]:
sample = tips["tip"].dropna()
t_stat, p_value = stats.ttest_1samp(sample, 3)
t_stat, p_value

(np.float64(-0.019432641422916876), np.float64(0.9845119176410544))

In [ ]:
print(p_value)

0.9845119176410544


In [ ]:
print(t_stat)

-0.019432641422916876


In [ ]:
# Derajat kebebasan
df = len(sample) - 1

# t-tabel pada alpha = 0.05 (two-tailed)
t_table = stats.t.ppf(1-0.025, df)

print("t-kritis: ", t_table)

t-kritis:  1.9697743954258793


* Hasil Uji One-Sample t-test\
t-statistic = -0.019\
t-kritis = 1.967\
p-value = 0.985
* Interpretasi:
  * H₀: rata-rata tip = \$3.
  * Karena |t-statistic| = 0.19 lebih kecil daripada t-kritis = -1.967, maka gagal menolak H₀
  * Dengan p-value = 0.985 (> 0.05), gagal menolak H₀.
  * Artinya: tidak ada bukti signifikan bahwa rata-rata tip berbeda dari \$3.
* Kesimpulan\
Berdasarkan uji One-Sample t-test, rata-rata tip pelanggan tidak berbeda signifikan dari \$3. Dengan kata lain, data mendukung bahwa \$3 merupakan nilai yang wajar untuk rata-rata tip.

# Independent Samples t-test
* Pertanyaan Penelitian\
Apakah terdapat perbedaan rata-rata tip antara pelanggan pria dan wanita?
* Uji Parametrik: Independent Samples t-test
* Tujuan: Menguji apakah rata-rata dua kelompok independen berbeda secara signifikan.
* Hipotesis:
  
  H₀: μₘₐₗₑ = μfₑₘₐₗₑ (rata-rata tip pria sama dengan wanita)

  H₁: μₘₐₗₑ ≠ μfₑₘₐₗₑ (rata-rata tip pria berbeda dari wanita)

In [ ]:
male = tips[tips["sex"]=="Male"]["tip"].dropna()
female = tips[tips["sex"]=="Female"]["tip"].dropna()

t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)
t_stat, p_value

(np.float64(1.489536377092501), np.float64(0.13780683808650296))

In [ ]:
print(t_stat)

1.489536377092501


In [ ]:
print(p_value)

0.13780683808650296


In [ ]:
# Derajat kebebasan (Welch–Satterthwaite)
df = (male.var()/male.count() + female.var()/female.count())**2 / (
    ((male.var()/male.count())**2)/(male.count()-1) +
    ((female.var()/female.count())**2)/(female.count()-1)
)

# Nilai t-tabel pada alpha = 0.05 (two-tailed)
t_table = stats.t.ppf(1-0.025, df)

print("t-kritis: ",t_table)

t-kritis:  1.9710225545069828


* Hasil Uji Independent Samples t-test
  
  t-statistic = 1.490

  t-kritis = 1.971
  
  p-value = 0.138
* Interpretasi:
  * H₀: tidak ada perbedaan rata-rata tip antara pria dan wanita.
  * Karena t-statistic = 1.490 lebih kecil dari t-kritis = 1.971 pada α=0.05, maka gagal tolak H₀.
  * Dengan p-value = 0.138 (> 0.05), gagal menolak H₀.
  * Artinya: tidak ada bukti signifikan bahwa rata-rata tip pria dan wanita berbeda.
* Kesimpulan\
Berdasarkan uji Independent Samples t-test, rata-rata tip antara pria dan wanita tidak berbeda signifikan. Dengan demikian, jenis kelamin pelanggan tidak memengaruhi rata-rata tip secara statistik.

# Paired Samples t-test
* Pertanyaan Penelitian\
Apakah terdapat perbedaan signifikan antara nilai total_bill dan tip (setelah disejajarkan per transaksi)?
* Uji Parametrik: Paired Samples t-test
* Tujuan: Membandingkan rata-rata dua pengukuran yang saling berpasangan (misalnya sebelum–sesudah, atau dua variabel terkait pada individu yang sama).
* Hipotesis:

  H₀: μᵦᵢₗₗ = μₜᵢₚ (tidak ada perbedaan rata-rata antara total_bill dan tip)

  H₁: μᵦᵢₗₗ ≠ μₜᵢₚ (ada perbedaan rata-rata antara total_bill dan tip)

In [ ]:
bill = tips["total_bill"].dropna()
tip = tips["tip"].dropna().iloc[:len(bill)]

t_stat, p_value = stats.ttest_rel(bill, tip)
t_stat, p_value

(np.float64(32.646504518298634), np.float64(8.020018605020848e-91))

In [ ]:
print(t_stat)

32.646504518298634


In [ ]:
print(p_value)

8.020018605020848e-91


In [ ]:
# Derajat kebebasan (df = n - 1)
df = len(bill) - 1

# Nilai t kritis untuk alpha = 0.05 (dua sisi)
alpha = 0.05
t_critical = stats.t.ppf(1 - alpha/2, df)
print("t-critical (0.05, two-tailed):", t_critical)

t-critical (0.05, two-tailed): 1.9697743954258793


* Hasil Uji Paired Samples t-test\
  t-statistic = 32.647
  
  t-kritis = 1.969

  p-value = 8.02e-91
* Interpretasi:
  * H₀: rata-rata total_bill sama dengan rata-rata tip.
  * Karena t-statistic = 32.647 jauh lebih besar daripada t-kritis = 1.969 pada α=0.05, maka tolak H₀.
  * Dengan p-value < 0.05, tolak H₀.
  * Artinya: terdapat perbedaan yang sangat signifikan antara rata-rata total_bill dan tip.
* Kesimpulan\
Hasil uji menunjukkan bahwa rata-rata total_bill dan tip berbeda secara signifikan. Hal ini wajar, karena jumlah total tagihan (total_bill) secara sistematis lebih besar dibandingkan tip yang diberikan.

# ANOVA (One-Way)
* Pertanyaan Penelitian\
Apakah terdapat perbedaan rata-rata panjang sepal (sepal length) antar spesies bunga Iris (setosa, versicolor, virginica)?
* Uji Parametrik: One-Way ANOVA
* Tujuan: Menguji apakah rata-rata dari lebih dari dua kelompok berbeda secara signifikan.
* Hipotesis:
  
  H₀: μₛₑₜₒₛₐ = μᵥₑᵣₛᵢcₒₗₒᵣ = μᵥᵢᵣgᵢₙᵢcₐ (tidak ada perbedaan rata-rata antar spesies).

  H₁: minimal ada satu spesies dengan rata-rata berbeda.

In [ ]:
anova = ols("sepal_length ~ species", data=iris).fit()
anova_table = sm.stats.anova_lm(anova, typ=2)
anova_table

,sum_sq,df,F,PR(>F)
species,63.212133,2.0,119.264502,1.669669e-31
Residual,38.956200,147.0,NaN,NaN


In [ ]:
# Contoh: ANOVA One-Way pada iris
alpha = 0.05
df1 = 2   # derajat bebas antar kelompok (k - 1)
df2 = 147 # derajat bebas dalam kelompok (N - k)

# Nilai F kritis (tabel F)
f_crit = stats.f.ppf(1 - alpha, df1, df2)
print("F kritis (alpha=0.05):", f_crit)

F kritis (alpha=0.05): 3.057620651649394


* Hasil Uji ANOVA (One-Way)

  F-statistic = 119.265

  F-kritis (α=0.05, df1=2, df2=147) = 3.057
  
  p-value = 1.67e-31

* Interpretasi:
  * H₀: tidak ada perbedaan rata-rata panjang sepal antar spesies.
  * Karena F-statistic = 119.265 > F-kritis = 3.057, tolak H₀.
  * Dengan p-value < 0.05, tolak H₀.
  * Artinya: terdapat perbedaan signifikan rata-rata panjang sepal pada minimal satu spesies Iris.
* Kesimpulan\
Berdasarkan uji One-Way ANOVA, panjang sepal bunga Iris berbeda signifikan antar spesies. Untuk mengetahui spesies mana yang berbeda secara spesifik, diperlukan uji lanjut (post-hoc test, misalnya Tukey HSD).

# Pearson Correlation
* Pertanyaan Penelitian\
Apakah terdapat hubungan linear yang signifikan antara panjang paruh (bill length) dan panjang sayap (flipper length) pada penguin?
* Uji Parametrik: Korelasi Pearson
* Tujuan: Mengukur kekuatan dan arah hubungan linear antara dua variabel kuantitatif.
* Hipotesis:
  
  H₀: ρ = 0 (tidak ada korelasi linear antara bill length dan flipper length).
  
  H₁: ρ ≠ 0 (ada korelasi linear signifikan antara bill length dan flipper length).

In [ ]:
penguins_clean = penguins.dropna(subset=["bill_length_mm","flipper_length_mm"])
corr, p_value = stats.pearsonr(penguins_clean["bill_length_mm"], penguins_clean["flipper_length_mm"])
corr, p_value

(np.float64(0.6530956386670859), np.float64(7.211340708097467e-42))

In [ ]:
print(corr)

0.6530956386670859


In [ ]:
print(p_value)

7.211340708097467e-42


* Hasil Uji Korelasi Pearson

  r (korelasi) = 0.653

  p-value = 7.21e-42

* Interpretasi:
  * Nilai korelasi r = 0.653 menunjukkan hubungan positif kuat antara panjang paruh dan panjang sayap.
  * Dengan p-value < 0.05, tolak H₀.
  * Artinya: terdapat korelasi linear signifikan antara kedua variabel.
* Kesimpulan\
Terdapat hubungan positif yang kuat antara panjang paruh dan panjang sayap pada penguin. Semakin panjang paruh, cenderung semakin panjang pula sayapnya.

# MANOVA (Multivariate ANOVA)
* Pertanyaan Penelitian\
Apakah terdapat perbedaan signifikan pada panjang paruh (bill length) dan panjang sayap (flipper length) penguin berdasarkan spesiesnya (Adelie, Chinstrap, Gentoo)?
* Uji Parametrik: MANOVA (Multivariate ANOVA)
* Tujuan: Menguji perbedaan rata-rata multivariat antar kelompok, yaitu apakah variabel independen (species) berpengaruh secara simultan terhadap beberapa variabel dependen (bill length dan flipper length).
* Hipotesis:

  H₀: Tidak ada perbedaan multivariat rata-rata bill length dan flipper length antar spesies.

  H₁: Ada perbedaan multivariat rata-rata pada minimal satu spesies.

In [ ]:
penguins_clean = penguins.dropna(subset=["bill_length_mm","flipper_length_mm","species"])
maov = MANOVA.from_formula("bill_length_mm + flipper_length_mm ~ species", data=penguins_clean)
print(maov.mv_test())

                    Multivariate linear model
                                                                  
------------------------------------------------------------------
       Intercept         Value   Num DF  Den DF   F Value   Pr > F
------------------------------------------------------------------
          Wilks' lambda   0.0028 2.0000 329.0000 59140.0033 0.0000
         Pillai's trace   0.9972 2.0000 329.0000 59140.0033 0.0000
 Hotelling-Lawley trace 359.5137 2.0000 329.0000 59140.0033 0.0000
    Roy's greatest root 359.5137 2.0000 329.0000 59140.0033 0.0000
------------------------------------------------------------------
                                                                  
------------------------------------------------------------------
            species         Value  Num DF  Den DF  F Value  Pr > F
------------------------------------------------------------------
              Wilks' lambda 0.0878 4.0000 658.0000 390.5566 0.0000
             Pil

* Hasil Uji MANOVA\
  Ringkasan hasil untuk faktor species:
  * Wilks’ Lambda = 0.0878, F(4, 658) = 390.56, p < 0.001
  * Pillai’s Trace = 1.3812, F(4, 660) = 368.29, p < 0.001
  * Hotelling-Lawley Trace = 5.0452, F ≈ 414.55, p < 0.001
  * Roy’s Greatest Root = 3.5342, F ≈ 583.15, p < 0.001
* Interpretasi:
  * Semua uji multivariat (Wilks, Pillai, Hotelling, Roy) menghasilkan p < 0.05.
  * Maka, tolak H₀ → terdapat perbedaan signifikan multivariat rata-rata bill length dan flipper length antar spesies.
* Kesimpulan\
Spesies penguin berpengaruh signifikan terhadap ukuran tubuh (panjang paruh dan panjang sayap). Dengan kata lain, spesies penguin memiliki profil morfologi yang berbeda secara statistik. Analisis lanjutan dapat dilakukan dengan uji post-hoc (misalnya pairwise MANOVA atau analisis univariat ANOVA per variabel).

# Penutup
Melalui serangkaian uji statistik parametrik yang telah dilakukan, portofolio ini menunjukkan bagaimana teori statistik dapat diterapkan secara nyata menggunakan Python.

Beberapa poin penting yang dapat disimpulkan:
* Asumsi Statistik perlu diuji sebelum melakukan analisis utama (normalitas & homogenitas).
* Uji t bermanfaat untuk perbandingan rata-rata sederhana, baik satu sampel, dua sampel independen, maupun pasangan berulang.
* ANOVA memungkinkan analisis perbedaan rata-rata lebih dari dua kelompok, sementara MANOVA memperluasnya ke multivariat.
* Korelasi Pearson memberi wawasan tentang hubungan linear antar variabel.
* Uji Chi-Square menjadi alternatif penting ketika berhadapan dengan data kategorikal atau ingin melihat kesesuaian distribusi dengan teori.
* Hasil analisis selalu perlu diinterpretasikan dalam konteks praktis, bukan hanya nilai p semata.

Portofolio ini diharapkan menjadi referensi yang tidak hanya menampilkan keterampilan teknis dalam Python, tetapi juga menekankan kemampuan berpikir kritis dan analitis dalam membaca hasil uji statistik. Dengan demikian, portofolio ini dapat menjadi pijakan yang kuat bagi penerapan statistik dalam berbagai bidang, mulai dari riset akademik hingga pengambilan keputusan strategis di industri.

# Thank You